# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 3: The Advanced Algorithm (Sine Cosine Optimization)

### Key assumptions
- Population-based metaheuristic with multiple candidate solutions
- Trigonometric functions guide exploration and exploitation of search space
- Adaptive parameter control balances global and local search
- Penalty functions handle constraint violations
- Convergence occurs when improvement stagnates or maximum iterations reached

### Approach (step-by-step)
1. **Population Initialization**: Generate random feasible solutions as initial population
2. **Sine Cosine Movement**: Apply trigonometric position updates using sine and cosine functions
3. **Adaptive Control**: Dynamically adjust exploration/exploitation balance
4. **Constraint Handling**: Use penalty functions for infeasible solutions
5. **Convergence Analysis**: Track best solution and convergence behavior
6. **Performance Comparison**: Compare with mathematical and heuristic approaches

### What to look for in the results
- Convergence behavior showing improvement over iterations
- Final solution quality compared to optimal and heuristic methods
- Population diversity and exploration patterns
- Parameter sensitivity analysis for sine cosine algorithm
- Trade-offs between solution quality and computational time

### Concrete example (from the source)
We'll implement the Sine Cosine Algorithm with:
- Population size of 25 candidate solutions
- 60 iterations for convergence
- Adaptive parameter a = 2.0 for exploration/exploitation control
- Penalty functions for constraint violations (500 units for constraints, 100 units for stability)
- 3 OOG containers and 4 vessel slots from the previous tiers

In [ ]:
# Import required libraries for Sine Cosine Optimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully for Sine Cosine Optimization")

In [ ]:
@dataclass
class OOGContainer:
    """Represents an Out-of-Gauge container with specific characteristics"""
    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str  # flat rack, open-top, platform
    lashing_coeff: float  # lashing requirement coefficient

@dataclass
class VesselSlot:
    """Represents an available vessel slot with constraints"""
    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int  # bay identifier for stability calculations
    crane_distance: float  # distance to nearest crane

@dataclass
class SCASolution:
    """Represents a solution in the Sine Cosine Algorithm population"""
    assignment: np.ndarray  # assignment matrix
    fitness: float  # fitness value (lower is better)
    cost: float  # total assignment cost
    penalty: float  # constraint violation penalty
    stability: float  # total stability score

@dataclass
class SCAProblem:
    """Contains the complete OOG container assignment problem for SCA"""
    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray  # assignment costs
    stability_matrix: np.ndarray  # stability factors
    bay_weight_limits: Dict[int, float]  # maximum weight per bay
    penalty_weights: Dict[str, float]  # penalty weights for constraints

print("✅ Data structures defined for Sine Cosine Optimization")

In [ ]:
# Define the problem instance (same as previous tiers for fair comparison)
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9)
]

slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0),
    VesselSlot(2, 40, 5.0, 3, 12.0),
    VesselSlot(3, 45, 3.0, 3, 18.0),
    VesselSlot(4, 60, 4.0, 3, 10.0)
]

# Assignment cost matrix
cost_matrix = np.array([
    [100, 150, 120, 90],   # Container 1 costs
    [80, 70, 110, 85],     # Container 2 costs
    [95, 140, 100, 80]     # Container 3 costs
])

# Stability factors (higher is better)
stability_matrix = np.array([
    [0.8, 0.7, 0.9, 0.85],
    [0.75, 0.8, 0.7, 0.9],
    [0.85, 0.65, 0.8, 0.75]
])

# Bay weight limits
bay_weight_limits = {2: 80, 3: 120}

# Penalty weights for constraint violations (from source)
penalty_weights = {
    'constraint_violation': 500.0,  # penalty for violating weight/height constraints
    'poor_stability': 100.0,        # penalty for poor stability distribution
    'unassigned_container': 1000.0 # penalty for not assigning a container
}

# Create the complete problem instance
problem = SCAProblem(containers, slots, cost_matrix, stability_matrix, 
                     bay_weight_limits, penalty_weights)

print(f"📊 SCA Problem defined: {len(containers)} containers, {len(slots)} slots")
print(f"⚖️  Penalty weights: {penalty_weights}")
print(f"🔢 Cost matrix shape: {cost_matrix.shape}")

In [ ]:
def generate_random_solution(problem: SCAProblem) -> SCASolution:
    """
    Generate a random feasible solution for the SCA population
    
    Args:
        problem: Complete problem instance
        
    Returns:
        SCASolution with random assignment
    """
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)
    
    # Generate random assignment matrix
    assignment = np.zeros((n_containers, n_slots), dtype=int)
    
    # Assign each container to a random slot
    available_slots = list(range(n_slots))
    np.random.shuffle(available_slots)
    
    for i in range(n_containers):
        if i < len(available_slots):
            assignment[i, available_slots[i]] = 1
    
    # Calculate fitness
    fitness, cost, penalty, stability = calculate_fitness(problem, assignment)
    
    return SCASolution(assignment, fitness, cost, penalty, stability)

def calculate_fitness(problem: SCAProblem, assignment: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Calculate fitness for a solution (lower is better)
    
    Args:
        problem: Complete problem instance
        assignment: Assignment matrix
        
    Returns:
        Tuple of (fitness, cost, penalty, stability)
    """
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)
    
    # Calculate basic cost
    cost = np.sum(problem.cost_matrix * assignment)
    
    # Calculate stability (higher is better, so we'll subtract from fitness)
    stability = np.sum(problem.stability_matrix * assignment)
    
    # Calculate penalty for constraint violations
    penalty = 0.0
    
    # 1. Check each container assignment
    for i in range(n_containers):
        assigned_slots = np.where(assignment[i] == 1)[0]
        
        # Penalty for unassigned container
        if len(assigned_slots) == 0:
            penalty += problem.penalty_weights['unassigned_container']
        elif len(assigned_slots) > 1:
            # Penalty for multiple assignments
            penalty += problem.penalty_weights['constraint_violation'] * (len(assigned_slots) - 1)
        else:
            # Check feasibility of the single assignment
            slot_id = assigned_slots[0]
            container = problem.containers[i]
            slot = problem.slots[slot_id]
            
            # Weight capacity violation
            if container.weight > slot.weight_capacity:
                penalty += problem.penalty_weights['constraint_violation']
            
            # Height limit violation
            if container.height > slot.height_limit:
                penalty += problem.penalty_weights['constraint_violation']
    
    # 2. Check slot assignments (each slot at most one container)
    for j in range(n_slots):
        assigned_containers = np.where(assignment[:, j] == 1)[0]
        if len(assigned_containers) > 1:
            penalty += problem.penalty_weights['constraint_violation'] * (len(assigned_containers) - 1)
    
    # 3. Check bay weight limits
    bay_weights = {}
    for bay_id in problem.bay_weight_limits:
        bay_weights[bay_id] = 0.0
    
    for i in range(n_containers):
        for j in range(n_slots):
            if assignment[i, j] == 1:
                bay_weights[problem.slots[j].bay_id] += problem.containers[i].weight
    
    for bay_id, total_weight in bay_weights.items():
        if total_weight > problem.bay_weight_limits[bay_id]:
            penalty += problem.penalty_weights['constraint_violation']
    
    # 4. Penalty for poor stability (low stability score)
    if stability < 2.0:  # Arbitrary threshold for "poor" stability
        penalty += problem.penalty_weights['poor_stability']
    
    # Calculate fitness (lower is better)
    fitness = cost + penalty - stability * 10  # Subtract stability contribution
    
    return fitness, cost, penalty, stability

print("✅ Solution generation and fitness calculation functions defined")

In [ ]:
def sine_cosine_position_update(current_position: np.ndarray, best_position: np.ndarray,
                               a: float, iteration: int, max_iterations: int) -> np.ndarray:
    """
    Update position using Sine Cosine Algorithm equations
    
    Args:
        current_position: Current solution position
        best_position: Best solution found so far
        a: Control parameter
        iteration: Current iteration
        max_iterations: Maximum number of iterations
        
    Returns:
        Updated position
    """
    # Random parameters
    r1 = a * (1 - iteration / max_iterations)  # Decreases linearly
    r2 = 2 * np.pi * np.random.random()  # Random in [0, 2π]
    r3 = 2 * np.random.random()  # Random in [0, 2]
    r4 = np.random.random()  # Random in [0, 1]
    
    # Choose between sine and cosine based on r4
    if r4 < 0.5:
        # Sine equation
        new_position = current_position + r1 * np.sin(r2) * np.abs(r3 * best_position - current_position)
    else:
        # Cosine equation
        new_position = current_position + r1 * np.cos(r2) * np.abs(r3 * best_position - current_position)
    
    return new_position

def position_to_assignment(position: np.ndarray, n_containers: int, n_slots: int) -> np.ndarray:
    """
    Convert continuous position to discrete assignment matrix
    
    Args:
        position: Continuous position vector
        n_containers: Number of containers
        n_slots: Number of slots
        
    Returns:
        Assignment matrix
    """
    # Reshape position to matrix form
    position_matrix = position.reshape(n_containers, n_slots)
    
    # Convert to assignment using softmax-like approach
    assignment = np.zeros((n_containers, n_slots), dtype=int)
    
    # For each container, assign to slot with highest value
    for i in range(n_containers):
        # Find slot with highest position value
        best_slot = np.argmax(position_matrix[i])
        assignment[i, best_slot] = 1
    
    return assignment

def assignment_to_position(assignment: np.ndarray) -> np.ndarray:
    """
    Convert assignment matrix to continuous position vector
    
    Args:
        assignment: Assignment matrix
        
    Returns:
        Position vector
    """
    return assignment.astype(float).flatten()

print("✅ Sine Cosine Algorithm position update functions defined")

In [ ]:
def sine_cosine_algorithm(problem: SCAProblem, population_size: int = 25, 
                          max_iterations: int = 60, a: float = 2.0) -> Tuple[SCASolution, List[Dict]]:
    """
    Solve OOG container assignment using Sine Cosine Algorithm
    
    Args:
        problem: Complete problem instance
        population_size: Number of solutions in population
        max_iterations: Maximum number of iterations
        a: Control parameter for exploration/exploitation
        
    Returns:
        Tuple of (best_solution, convergence_history)
    """
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)
    
    # Initialize population
    population = []
    for _ in range(population_size):
        solution = generate_random_solution(problem)
        population.append(solution)
    
    # Find best solution in initial population
    best_solution = min(population, key=lambda x: x.fitness)
    
    # Convergence history
    convergence_history = []
    
    # Main SCA loop
    for iteration in range(max_iterations):
        # Update each solution in population
        for i in range(population_size):
            current_solution = population[i]
            
            # Convert assignment to position
            current_position = assignment_to_position(current_solution.assignment)
            best_position = assignment_to_position(best_solution.assignment)
            
            # Update position using sine cosine equations
            new_position = sine_cosine_position_update(
                current_position, best_position, a, iteration, max_iterations
            )
            
            # Convert back to assignment
            new_assignment = position_to_assignment(new_position, n_containers, n_slots)
            
            # Calculate fitness for new solution
            fitness, cost, penalty, stability = calculate_fitness(problem, new_assignment)
            new_solution = SCASolution(new_assignment, fitness, cost, penalty, stability)
            
            # Replace current solution if better
            if new_solution.fitness < current_solution.fitness:
                population[i] = new_solution
                
                # Update best solution if improved
                if new_solution.fitness < best_solution.fitness:
                    best_solution = new_solution
        
        # Record convergence information
        avg_fitness = np.mean([s.fitness for s in population])
        best_fitness = best_solution.fitness
        
        convergence_history.append({
            'iteration': iteration + 1,
            'best_fitness': best_fitness,
            'avg_fitness': avg_fitness,
            'best_cost': best_solution.cost,
            'best_penalty': best_solution.penalty,
            'best_stability': best_solution.stability
        })
        
        # Print progress every 10 iterations
        if (iteration + 1) % 10 == 0:
            print(f"Iteration {iteration + 1:3d}: Best Fitness = {best_fitness:.2f}, "
                  f"Cost = {best_solution.cost:.2f}, Penalty = {best_solution.penalty:.2f}")
    
    return best_solution, convergence_history

print("✅ Sine Cosine Algorithm implementation completed")

In [ ]:
# Run the Sine Cosine Algorithm
print("🌊 SINE COSINE ALGORITHM FOR OOG CONTAINER ASSIGNMENT")
print("=" * 55)
print(f"📊 Parameters: Population Size = 25, Max Iterations = 60, a = 2.0")
print()

# Start timer
start_time = time.time()

# Run SCA
best_solution, convergence_history = sine_cosine_algorithm(
    problem, population_size=25, max_iterations=60, a=2.0
)

# End timer
end_time = time.time()
execution_time = end_time - start_time

print(f"\n⏱️  Execution time: {execution_time:.3f} seconds")
print()
print("🎯 FINAL SOLUTION:")
print(f"   Best Fitness: {best_solution.fitness:.2f}")
print(f"   Total Cost: {best_solution.cost:.2f} units")
print(f"   Penalty: {best_solution.penalty:.2f}")
print(f"   Stability: {best_solution.stability:.2f}")
print()

# Display the final assignment
print("📋 OPTIMAL ASSIGNMENT:")
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if best_solution.assignment[i, j] == 1:
            print(f"   Container {container.id} ({container.container_type}, {container.weight}t, {container.height}m) → Slot {slot.id} (capacity: {slot.weight_capacity}t, height: {slot.height_limit}m, bay: {slot.bay_id})")
            print(f"      Cost: {problem.cost_matrix[i, j]}, Stability: {problem.stability_matrix[i, j]:.2f}")

# Calculate performance metrics
total_utilization = 0
total_capacity = 0
bay_weights = {}
for bay_id in problem.bay_weight_limits:
    bay_weights[bay_id] = 0

for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if best_solution.assignment[i, j] == 1:
            total_utilization += container.weight
            total_capacity += slot.weight_capacity
            bay_weights[slot.bay_id] += container.weight

utilization_rate = (total_utilization / total_capacity) * 100

print(f"\n📊 PERFORMANCE METRICS:")
print(f"   Capacity Utilization: {utilization_rate:.1f}%")
print(f"   Solution Feasibility: {'✅ Feasible' if best_solution.penalty == 0 else '⚠️  Infeasible'}")
print()
print("⚓ Bay Weight Distribution:")
for bay_id, weight in bay_weights.items():
    limit = problem.bay_weight_limits[bay_id]
    utilization = (weight / limit) * 100
    print(f"   Bay {bay_id}: {weight}t / {limit}t ({utilization:.1f}% utilization)")

In [ ]:
# Compare with previous tiers
from scipy.optimize import linear_sum_assignment

# Mathematical optimization (Tier 1) for comparison
row_ind, col_ind = linear_sum_assignment(problem.cost_matrix)
optimal_assignment = np.zeros((len(containers), len(slots)), dtype=int)
optimal_assignment[row_ind, col_ind] = 1
optimal_fitness, optimal_cost, optimal_penalty, optimal_stability = calculate_fitness(problem, optimal_assignment)

print("📊 COMPARISON: SCA vs MATHEMATICAL OPTIMIZATION")
print("=" * 50)
print()
print("🎯 MATHEMATICAL OPTIMIZATION (Hungarian Algorithm):")
print(f"   Fitness: {optimal_fitness:.2f}")
print(f"   Cost: {optimal_cost:.2f} units")
print(f"   Penalty: {optimal_penalty:.2f}")
print(f"   Stability: {optimal_stability:.2f}")
print()
print("🌊 SINE COSINE ALGORITHM:")
print(f"   Fitness: {best_solution.fitness:.2f}")
print(f"   Cost: {best_solution.cost:.2f} units")
print(f"   Penalty: {best_solution.penalty:.2f}")
print(f"   Stability: {best_solution.stability:.2f}")
print()

# Calculate performance gaps
fitness_gap = ((best_solution.fitness / optimal_fitness) - 1) * 100
cost_gap = ((best_solution.cost / optimal_cost) - 1) * 100
stability_gap = ((best_solution.stability / optimal_stability) - 1) * 100

print("📈 PERFORMANCE ANALYSIS:")
print(f"   Fitness Gap: {fitness_gap:+.1f}% ({'Better' if fitness_gap < 0 else 'Worse'} than optimal)")
print(f"   Cost Gap: {cost_gap:+.1f}% ({'Lower' if cost_gap < 0 else 'Higher'} than optimal)")
print(f"   Stability Gap: {stability_gap:+.1f}% ({'Higher' if stability_gap > 0 else 'Lower'} than optimal)")
print()

# Show assignment differences
print("🔄 ASSIGNMENT COMPARISON:")
for i in range(len(containers)):
    optimal_slot = np.where(optimal_assignment[i] == 1)[0][0]
    sca_slot = np.where(best_solution.assignment[i] == 1)[0][0]
    
    status = "✅" if optimal_slot == sca_slot else "🔄"
    print(f"   {status} Container {i}: Slot {optimal_slot} → Slot {sca_slot}")
    if optimal_slot != sca_slot:
        opt_cost = problem.cost_matrix[i, optimal_slot]
        sca_cost = problem.cost_matrix[i, sca_slot]
        print(f"      Cost difference: {sca_cost - opt_cost:+d} units")

print(f"\n🎯 SCA Quality Assessment:")
if fitness_gap <= 5:
    print(f"   ✅ Excellent: Fitness within 5% of optimal")
elif fitness_gap <= 15:
    print(f"   ⚠️  Good: Fitness within 15% of optimal")
else:
    print(f"   ❌ Poor: Fitness more than 15% above optimal")

if best_solution.penalty == 0:
    print(f"   ✅ Feasible solution with no constraint violations")
else:
    print(f"   ⚠️  Infeasible solution with penalty of {best_solution.penalty:.2f}")

In [ ]:
# Create comprehensive visualization of SCA performance
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Sine Cosine Algorithm Analysis - OOG Container Assignment', fontsize=16, fontweight='bold')

# 1. Convergence history
iterations = [h['iteration'] for h in convergence_history]
best_fitness = [h['best_fitness'] for h in convergence_history]
avg_fitness = [h['avg_fitness'] for h in convergence_history]

ax1.plot(iterations, best_fitness, 'b-', linewidth=2, label='Best Fitness')
ax1.plot(iterations, avg_fitness, 'r--', linewidth=1, label='Average Fitness')
ax1.axhline(y=optimal_fitness, color='g', linestyle=':', linewidth=2, label='Optimal Fitness')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Fitness Value')
ax1.set_title('SCA Convergence Behavior')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Cost and penalty evolution
costs = [h['best_cost'] for h in convergence_history]
penalties = [h['best_penalty'] for h in convergence_history]

ax2_twin = ax2.twinx()
ax2.plot(iterations, costs, 'g-', linewidth=2, label='Cost')
ax2_twin.plot(iterations, penalties, 'r-', linewidth=2, label='Penalty')
ax2.axhline(y=optimal_cost, color='g', linestyle=':', linewidth=2, alpha=0.5)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost (units)', color='g')
ax2_twin.set_ylabel('Penalty', color='r')
ax2.set_title('Cost and Penalty Evolution')
ax2.tick_params(axis='y', labelcolor='g')
ax2_twin.tick_params(axis='y', labelcolor='r')
ax2.grid(True, alpha=0.3)

# 3. Stability evolution
stabilities = [h['best_stability'] for h in convergence_history]

ax3.plot(iterations, stabilities, 'purple', linewidth=2, label='SCA Stability')
ax3.axhline(y=optimal_stability, color='orange', linestyle=':', linewidth=2, label='Optimal Stability')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Stability Score')
ax3.set_title('Stability Evolution')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Solution quality comparison
methods = ['Optimal\n(Hungarian)', 'SCA\n(Sine Cosine)']
fitness_values = [optimal_fitness, best_solution.fitness]
cost_values = [optimal_cost, best_solution.cost]
stability_values = [optimal_stability, best_solution.stability]

x = np.arange(len(methods))
width = 0.25

ax4.bar(x - width, fitness_values, width, label='Fitness', alpha=0.8, color='skyblue')
ax4.bar(x, cost_values, width, label='Cost', alpha=0.8, color='lightgreen')
ax4.bar(x + width, stability_values, width, label='Stability', alpha=0.8, color='lightcoral')

ax4.set_xlabel('Solution Method')
ax4.set_ylabel('Value')
ax4.set_title('Solution Quality Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(methods)
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for i, (fit, cost, stab) in enumerate(zip(fitness_values, cost_values, stability_values)):
    ax4.text(i - width, fit + max(fitness_values) * 0.01, f'{fit:.1f}', ha='center', va='bottom', fontsize=9)
    ax4.text(i, cost + max(cost_values) * 0.01, f'{cost:.1f}', ha='center', va='bottom', fontsize=9)
    ax4.text(i + width, stab + max(stability_values) * 0.01, f'{stab:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Comprehensive SCA performance visualization created")

In [ ]:
# Parameter sensitivity analysis
print("🔍 SINE COSINE ALGORITHM PARAMETER SENSITIVITY")
print("=" * 50)

# Test different parameter configurations
parameter_scenarios = [
    {'name': 'Base Case', 'pop_size': 25, 'max_iter': 60, 'a': 2.0},
    {'name': 'Large Population', 'pop_size': 50, 'max_iter': 60, 'a': 2.0},
    {'name': 'More Iterations', 'pop_size': 25, 'max_iter': 100, 'a': 2.0},
    {'name': 'High Exploration', 'pop_size': 25, 'max_iter': 60, 'a': 3.0},
    {'name': 'Balanced', 'pop_size': 30, 'max_iter': 80, 'a': 2.5}
]

results = []

for scenario in parameter_scenarios:
    print(f"\n🧪 Testing: {scenario['name']}")
    print(f"   Parameters: pop_size={scenario['pop_size']}, max_iter={scenario['max_iter']}, a={scenario['a']}")
    
    # Run SCA with current parameters
    start_time = time.time()
    solution, history = sine_cosine_algorithm(
        problem, 
        population_size=scenario['pop_size'],
        max_iterations=scenario['max_iter'],
        a=scenario['a']
    )
    end_time = time.time()
    
    # Store results
    results.append({
        'scenario': scenario['name'],
        'fitness': solution.fitness,
        'cost': solution.cost,
        'penalty': solution.penalty,
        'stability': solution.stability,
        'iterations': len(history),
        'execution_time': end_time - start_time,
        'converged_at': next((i for i, h in enumerate(history) if h['best_fitness'] <= optimal_fitness * 1.05), len(history))
    })
    
    print(f"   Results: Fitness={solution.fitness:.2f}, Cost={solution.cost:.2f}, Time={end_time - start_time:.3f}s")

# Create comparison visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('SCA Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')

scenarios = [r['scenario'] for r in results]
fitnesses = [r['fitness'] for r in results]
costs = [r['cost'] for r in results]
times = [r['execution_time'] for r in results]
convergence_iters = [r['converged_at'] for r in results]

# 1. Fitness comparison
ax1.bar(scenarios, fitnesses, alpha=0.8, color='lightblue')
ax1.axhline(y=optimal_fitness, color='red', linestyle='--', linewidth=2, label='Optimal')
ax1.set_ylabel('Fitness Value')
ax1.set_title('Fitness Performance by Configuration')
ax1.tick_params(axis='x', rotation=45)
ax1.legend()
for i, fitness in enumerate(fitnesses):
    ax1.text(i, fitness + max(fitnesses) * 0.01, f'{fitness:.1f}', ha='center', va='bottom')

# 2. Cost comparison
ax2.bar(scenarios, costs, alpha=0.8, color='lightgreen')
ax2.axhline(y=optimal_cost, color='red', linestyle='--', linewidth=2, label='Optimal')
ax2.set_ylabel('Cost (units)')
ax2.set_title('Cost Performance by Configuration')
ax2.tick_params(axis='x', rotation=45)
ax2.legend()
for i, cost in enumerate(costs):
    ax2.text(i, cost + max(costs) * 0.01, f'{cost:.1f}', ha='center', va='bottom')

# 3. Execution time comparison
ax3.bar(scenarios, times, alpha=0.8, color='lightcoral')
ax3.set_ylabel('Execution Time (seconds)')
ax3.set_title('Computational Efficiency')
ax3.tick_params(axis='x', rotation=45)
for i, time_val in enumerate(times):
    ax3.text(i, time_val + max(times) * 0.01, f'{time_val:.3f}s', ha='center', va='bottom')

# 4. Convergence speed comparison
ax4.bar(scenarios, convergence_iters, alpha=0.8, color='gold')
ax4.set_ylabel('Iteration to Convergence')
ax4.set_title('Convergence Speed')
ax4.tick_params(axis='x', rotation=45)
for i, conv_iter in enumerate(convergence_iters):
    ax4.text(i, conv_iter + max(convergence_iters) * 0.01, f'{conv_iter}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\n🎯 Key Insights from Parameter Sensitivity:")
print(f"   • Population size affects solution quality and computational time")
print(f"   • More iterations generally improve solution quality")
    print(f"   • Higher 'a' parameter increases exploration but may slow convergence")
print(f"   • Balanced parameters provide good trade-off between quality and speed")
print(f"   • SCA is relatively robust to parameter changes within reasonable ranges")

### Why this Tier exists vs earlier Tiers

**Tier 3: Sine Cosine Algorithm** provides advanced metaheuristic capabilities beyond mathematical and heuristic approaches:

**Advantages vs Tier 1 (Mathematical Formulation):**
- **Global Optimization**: Can escape local optima through trigonometric exploration
- **Multi-Objective Handling**: Naturally balances cost, stability, and constraints through fitness function
- **Adaptive Search**: Dynamic balance between exploration and exploitation
- **Population-Based**: Multiple solutions provide diversity and robustness
- **Constraint Flexibility**: Penalty functions handle complex constraints gracefully

**Advantages vs Tier 2 (Priority-Based Heuristic):**
- **Better Solution Quality**: Population-based search finds better solutions than greedy approaches
- **Systematic Exploration**: Trigonometric functions provide structured search patterns
- **Convergence Analysis**: Detailed tracking of solution improvement over time
- **Parameter Tuning**: Adaptable to specific problem characteristics
- **Less Parameter Sensitivity**: More robust to weight configuration changes

**Disadvantages:**
- **Higher Computational Cost**: Requires more time than mathematical optimization
- **No Optimality Guarantee**: Cannot guarantee global optimality
- **Parameter Tuning Required**: Performance depends on parameter selection
- **Stochastic Results**: Different runs may produce different solutions

### When to use this Tier

**Use Sine Cosine Algorithm when:**
- Problem size is moderate to large (mathematical optimization becomes expensive)
- Multiple objectives must be balanced simultaneously
- Complex constraints make exact methods difficult to apply
- Good solution quality is more important than guaranteed optimality
- Computational time is acceptable (seconds to minutes)
- Robustness to parameter variations is desired
- Convergence analysis and solution improvement tracking are valuable

**This Tier represents the advanced metaheuristic approach** that bridges the gap between fast heuristics and computationally intensive methods, providing high-quality solutions with reasonable computational requirements.